## PCoA / PERMANOVA Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from pathlib import Path
from skbio.stats.distance import DistanceMatrix, permanova

# Keep text editable in SVG and use Arial
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.unicode_minus'] = False

# Auto-detect input file
cands = [p for p in Path('.').glob('*PCOA.xlsx') if not p.name.startswith('~$')]
if len(cands) != 1:
    raise RuntimeError('Current folder must contain exactly one *PCOA.xlsx file')

infile = cands[0]
outfile = Path('functional_pcoa_minmax_bray_genus_centroid.svg')

# Load data
df = pd.read_excel(infile, sheet_name=0)
genus = df.iloc[:, 1].astype(str)
X = df.iloc[:, 4:].to_numpy(float)

# Min-Max scaling by feature
col_min = X.min(axis=0)
col_max = X.max(axis=0)
span = np.where((col_max - col_min) == 0, 1.0, (col_max - col_min))
X_scaled = (X - col_min) / span

# Bray-Curtis distance on Min-Max scaled matrix
D = squareform(pdist(X_scaled, metric='braycurtis'))

# Classical PCoA
n = D.shape[0]
J = np.eye(n) - np.ones((n, n)) / n
B = -0.5 * J @ (D ** 2) @ J
B = (B + B.T) / 2.0

eigvals, eigvecs = np.linalg.eigh(B)
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
pos = eigvals > 0
if pos.sum() < 2:
    raise RuntimeError('Fewer than 2 positive eigenvalues; cannot draw 2D PCoA')

coords = eigvecs[:, :2] * np.sqrt(eigvals[:2])
var_explained = eigvals[pos] / eigvals[pos].sum()

# PERMANOVA based on the same distance matrix
ids = [f's{i}' for i in range(n)]
dm = DistanceMatrix(D, ids=ids)
res = permanova(dm, grouping=np.asarray(genus, dtype=str), permutations=999)
F = float(res['test statistic'])
g = int(res['number of groups'])
R2 = (F * (g - 1)) / (F * (g - 1) + (n - g))
p_value = float(res['p-value'])

# Plot
fig, ax = plt.subplots(figsize=(13, 10), dpi=300)
groups = pd.unique(genus)
k = len(groups)

if k <= 20:
    cmap = plt.get_cmap('tab20', k)
else:
    cmap = plt.get_cmap('gist_ncar', k)
colors = [cmap(i) for i in range(k)]
color_map = dict(zip(groups, colors))

# Draw point-to-centroid lines and cache centroids
centroids = {}
for gname in groups:
    m = (genus == gname).to_numpy()
    pts = coords[m]
    if len(pts) == 0:
        continue
    c = color_map[gname]
    centroid = pts.mean(axis=0)
    centroids[gname] = centroid
    for p in pts:
        ax.plot([p[0], centroid[0]], [p[1], centroid[1]], color=c, alpha=0.20, linewidth=0.6, zorder=1)

# Sample points (with labels for legend)
for gname in groups:
    m = (genus == gname).to_numpy()
    ax.scatter(
        coords[m, 0], coords[m, 1],
        s=24, alpha=0.85, color=color_map[gname], linewidths=0,
        label=gname, zorder=2
    )

# Centroid points
for gname in groups:
    if gname not in centroids:
        continue
    c = color_map[gname]
    centroid = centroids[gname]
    ax.scatter(
        centroid[0], centroid[1], s=45, marker='o',
        facecolor=c, edgecolor='black', linewidth=0.8, zorder=5
    )

# Genus labels near centroids
x_rng = np.ptp(coords[:, 0]) if np.ptp(coords[:, 0]) > 0 else 1.0
y_rng = np.ptp(coords[:, 1]) if np.ptp(coords[:, 1]) > 0 else 1.0
for i, gname in enumerate(groups):
    if gname not in centroids:
        continue
    cx, cy = centroids[gname]
    angle = 2 * np.pi * i / max(1, k)
    dx = 0.012 * x_rng * np.cos(angle)
    dy = 0.012 * y_rng * np.sin(angle)
    ax.text(
        cx + dx, cy + dy, gname,
        fontsize=8.5, color='black',
        ha='center', va='center', zorder=6,
        bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor=color_map[gname], linewidth=0.6, alpha=0.78)
    )

# PERMANOVA stats in top-left of plotting area
stats_text = f'PERMANOVA\nR^2 = {R2:.3f}\np = {p_value:.3g}'
ax.text(
    0.02, 0.98, stats_text,
    transform=ax.transAxes,
    ha='left', va='top', fontsize=11,
    bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='black', linewidth=0.8, alpha=0.88),
    zorder=7
)

ax.set_xlabel(f'PCoA1 ({var_explained[0]*100:.2f}%)')
ax.set_ylabel(f'PCoA2 ({var_explained[1]*100:.2f}%)')
ax.set_title('Functional PCoA (Min-Max + Bray-Curtis, colored by genus)')
ax.axhline(0, color='grey', linewidth=0.6, alpha=0.5)
ax.axvline(0, color='grey', linewidth=0.6, alpha=0.5)
ax.grid(alpha=0.2, linestyle='--', linewidth=0.5)

# Keep legend even with in-plot genus labels
ax.legend(
    title='genus', bbox_to_anchor=(1.02, 1), loc='upper left',
    frameon=False, fontsize=8, title_fontsize=9, ncol=1
)

plt.tight_layout()
plt.savefig(outfile, format='svg', bbox_inches='tight')
plt.show()

print(f'Output file: {outfile.name}')
print(f'PCoA1: {var_explained[0]*100:.2f}%')
print(f'PCoA2: {var_explained[1]*100:.2f}%')
print(f'R^2: {R2:.6f}; p: {p_value:.6f}')


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova

# Reuse existing variables from previous cells when available.
# Expected: X_scaled (samples x features), genus (group labels)
if 'X_scaled' not in globals() or 'genus' not in globals():
    cands = [p for p in Path('.').glob('*PCOA.xlsx') if not p.name.startswith('~$')]
    if len(cands) != 1:
        raise RuntimeError('??????????? 1 ? *PCOA.xlsx ??')

    df = pd.read_excel(cands[0], sheet_name=0)
    genus = df.iloc[:, 1].astype(str)
    X = df.iloc[:, 4:].to_numpy(dtype=float)

    # Min-Max scaling by feature
    x_min = X.min(axis=0)
    x_max = X.max(axis=0)
    span = np.where((x_max - x_min) == 0, 1.0, (x_max - x_min))
    X_scaled = (X - x_min) / span

# Bray-Curtis distance matrix
D = squareform(pdist(np.asarray(X_scaled, dtype=float), metric='braycurtis'))
ids = [f's{i}' for i in range(D.shape[0])]
dm = DistanceMatrix(D, ids=ids)

groups = np.asarray(genus, dtype=str)
res = permanova(dm, grouping=groups, permutations=999)

# Compute R^2 from pseudo-F, n, and number of groups
F = float(res['test statistic'])
n = int(res['sample size'])
g = int(res['number of groups'])
R2 = (F * (g - 1)) / (F * (g - 1) + (n - g))
p_value = float(res['p-value'])

print(f'PERMANOVA pseudo-F: {F:.6f}')
print(f'PERMANOVA R^2: {R2:.6f}')
print(f'PERMANOVA p-value: {p_value:.6f}')
res
